# DeepFake Image Detection
**Team No: 06** | EfficientNet-B4 + Transfer Learning + Grad-CAM

---
### Members
| Name | Roll No |
|------|--------|
| Ananthan K | AM.SC.P2ARI25006 |
| Amrutha M | AM.SC.P2ARI25004 |
| Mohammed Aariff L | AM.SC.P2ARI25014 |

**Submitted to:** Akshara P Byju

## 1. Setup & Dependencies

In [ ]:
# Install required packages (run once)
!pip install -q torch torchvision efficientnet-pytorch kagglehub \
    scikit-learn matplotlib seaborn tqdm pandas pillow

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import transforms
from torch.utils.data import DataLoader

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')

## 2. Download Dataset via KaggleHub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download('manjilkarki/deepfake-and-real-images')
print('Path to dataset files:', path)

## 3. Explore Dataset

In [ ]:
from src.dataset import DeepFakeDataset
from torchvision import transforms

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

for split in ('train', 'val', 'test'):
    ds = DeepFakeDataset(path, split=split, transform=tf)
    print(f'{split:5s}: {len(ds):6,} images | {ds.class_distribution()}')

In [ ]:
# Visualise random samples
import random
from PIL import Image

train_ds = DeepFakeDataset(path, split='train',
                           transform=transforms.Compose([transforms.Resize((224,224)),
                                                         transforms.ToTensor()]))
label_names = {0: 'Real', 1: 'Fake'}

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for ax in axes.flat:
    idx = random.randint(0, len(train_ds)-1)
    img, lbl = train_ds[idx]
    ax.imshow(img.permute(1,2,0))
    ax.set_title(label_names[lbl], color='green' if lbl==0 else 'red', fontsize=9)
    ax.axis('off')
plt.suptitle('Random Training Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Model Architecture

In [ ]:
from src.model import DeepFakeDetector

model = DeepFakeDetector(dropout_rate=0.4)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total:,}')
print(f'Trainable parameters : {trainable:,}')
print(f'Frozen parameters    : {total - trainable:,}')

## 5. Train the Model

In [ ]:
from src.train import main, CONFIG

CONFIG['dataset_path'] = path
CONFIG['num_epochs']   = 20
CONFIG['batch_size']   = 32

main(CONFIG)

## 6. Plot Training History

In [ ]:
import json
from utils.metrics import plot_training_history

with open('outputs/history.json') as f:
    history = json.load(f)

plot_training_history(history, save_path='outputs/training_curves.png')
plt.show()

## 7. Confusion Matrix & Metrics

In [ ]:
with open('outputs/test_metrics.json') as f:
    metrics = json.load(f)

print('Test Metrics:')
for k, v in metrics.items():
    if k != 'cm':
        print(f'  {k:12s}: {v:.4f}')

from utils.metrics import plot_confusion_matrix
plot_confusion_matrix(metrics['cm'])
plt.show()

## 8. Grad-CAM Visualisations

In [ ]:
from utils.grad_cam import GradCAM, visualize_grad_cam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load('outputs/best_model.pth', map_location=device)
model  = DeepFakeDetector()
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_ds = DeepFakeDataset(path, split='test', transform=val_tf)

grad_cam = GradCAM(model, target_layer=model.backbone.features[-1])
visualize_grad_cam(grad_cam, test_ds, device, save_dir='outputs/grad_cam', num_samples=8)

# Display saved Grad-CAM images inline
import glob
saved = sorted(glob.glob('outputs/grad_cam/*.png'))[:4]
fig, axes = plt.subplots(1, len(saved), figsize=(18, 5))
for ax, p in zip(axes, saved):
    ax.imshow(plt.imread(p))
    ax.set_title(os.path.basename(p).replace('.png',''), fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Single Image Inference
Use this cell to test the model on any image.

In [ ]:
from src.predict import predict_image, get_transform, load_model

test_image_path = 'path/to/your/face.jpg'  # <-- Change this

model_inf = load_model('outputs/best_model.pth', device)
tf_inf    = get_transform(224)
result    = predict_image(test_image_path, model_inf, tf_inf, device)

print(f"Prediction : {result['label']}")
print(f"Fake prob  : {result['fake_prob']:.4f}")
print(f"Confidence : {result['confidence']:.4f}")

img = Image.open(test_image_path)
plt.imshow(img); plt.axis('off')
plt.title(f"{result['label']} (conf={result['confidence']:.2f})", fontsize=13)
plt.show()